# One Postgres, every process — the catalog and the broker in production

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/29-production/production.ipynb)

Built from [`cookbook/book/chapters/29-production/production.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/29-production/production.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook[postgres] @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book", server + "==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `[catalog.postgres]` · `[broker.postgres]` · `jammi.testing.LiveServer` ·
`get_server_info` · `add_source` · `generate_embeddings` · `search` · `register_topic` ·
`publish_topic` · `subscribe_collect` · **Theory:** one database as the system of record
every process reads and writes, and an append-only log it replays
(Kleppmann 2017; Kreps et al. 2011) · **Rail:** measurement (what one process wrote,
another reads back row for row; every event one process publishes, another receives).

Every chapter so far ran on the development default: a SQLite catalog file and an
in-process broker. That default is single-process by construction — one engine owns
the file. A deployment with more than one process (a server and a batch job, two
server replicas, a worker fleet) runs its catalog and its broker on **Postgres**
instead: the catalog's rows (sources, models, result tables, jobs) live in one
database every process shares, and the broker is Postgres `LISTEN`/`NOTIFY` over the
topics' own backing tables, which live in that same database. Nothing else changes —
the same binary, the same verbs; the deployment is configuration.

This chapter runs that deployment in one runtime: a Postgres installed with `pip`
(`pgserver`), a `jammi-server` process, and an embedded engine in this notebook's own
process, both configured onto the same database.

In [ ]:
import tempfile
import threading
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import jammi
import pgserver
import pyarrow as pa
from jammi.testing import LiveServer
from jammi_cookbook import contracts, fixtures

MODEL = fixtures.model("tiny_bert")
CORPUS = fixtures.url("tiny_corpus.parquet")
QUERY = "how does quantum computing work?"

work = Path(tempfile.mkdtemp())
postgres = pgserver.get_server(work / "pgdata")
ARTIFACTS = work / "artifacts"


def catalog_url(postgres) -> str:
    """The engine's URL for `postgres`, reached over its Unix socket: a named host,
    with the socket directory in `host` (the libpq empty-host form is refused)."""
    socket_dir = parse_qs(urlparse(postgres.get_uri()).query)["host"][0]
    return f"postgresql://postgres@localhost/postgres?host={socket_dir}"

## The configuration: one database, two roles

The catalog stanza names the database — here over its Unix socket, so the URL names a
host and carries the socket directory in `host`; the broker stanza names no URL of its
own, so it uses the catalog's — one Postgres serves both roles. `idle_poll_secs` bounds how
long a subscriber waits for a wake-up it might have missed.

In [ ]:
config = work / "jammi.toml"
config.write_text(f"""
[catalog.postgres]
url = "{catalog_url(postgres)}"

[broker.postgres]
idle_poll_secs = 1

[gpu]
device = -1
""")
print(config.read_text())

## Two processes, one catalog

The server process registers a source and embeds it. The embedded engine — a different
process, opened on the same configuration — sees the source, sees the embedding table,
and searches it: the rows it gets are the rows the server gets.

In [ ]:
def ranked(db) -> list[tuple[str, float]]:
    query = db.encode_query(model=MODEL, query=QUERY)
    hits = db.search("corpus", query=query, k=5).to_pylist()
    return [(hit["_row_id"], round(hit["similarity"], 6)) for hit in hits]


with LiveServer(ARTIFACTS, env={"JAMMI_CONFIG": str(config)}) as server, \
        jammi.connect(server.endpoint) as remote:
    remote.add_source("corpus", url=CORPUS, format="parquet")
    table = remote.generate_embeddings(
        source="corpus", model=MODEL, columns=["content"], key="id", modality="text")
    served = ranked(remote)
    server_broker = remote.get_server_info()["broker"]

    with jammi.connect(f"file://{ARTIFACTS}", config=str(config)) as local:
        seen = [t["table_name"] for s in local.list_sources() for t in s["result_tables"]]
        embedded = ranked(local)
        local_broker = local.get_server_info()["broker"]

print(f"broker: server={server_broker}, embedded={local_broker}")
print(f"the embedded engine sees the server's table: {table in seen}")
for (key, score), (key_e, score_e) in zip(served, embedded):
    print(f"  {key:>8} {score:.6f}   {key_e:>8} {score_e:.6f}")

In [ ]:
assert server_broker == local_broker == "postgres"
assert table in seen
assert served == embedded
contracts.assert_close("production.shared_catalog_rows", len(embedded))

The embedded engine did not copy anything and was not told about the table: it read
the same `result_tables` row from the same database, and the same Parquet the row
names, from the same result root.

## Events across processes

A topic's backing table is a table in the catalog database, and the Postgres broker
wakes every subscriber — in any process — when a publish commits. So one process
publishes and another receives. Here a subscriber in the server process waits for
three batches while the embedded engine publishes them.

In [ ]:
SCHEMA = pa.schema([("op", pa.string()), ("key", pa.string())])
batches = [
    pa.table({"op": ["add", "add"], "key": ["a", "b"]}, schema=SCHEMA),
    pa.table({"op": ["update"], "key": ["a"]}, schema=SCHEMA),
    pa.table({"op": ["delete", "add"], "key": ["b", "c"]}, schema=SCHEMA),
]

with LiveServer(ARTIFACTS, env={"JAMMI_CONFIG": str(config)}) as server, \
        jammi.connect(server.endpoint) as remote:
    remote.register_topic("changes", schema=SCHEMA)
    received = {}

    def subscribe() -> None:
        received["live"] = remote.subscribe_collect(
            "changes", from_offset=0, replay_only=False, max_batches=len(batches))

    subscriber = threading.Thread(target=subscribe)
    subscriber.start()
    with jammi.connect(f"file://{ARTIFACTS}", config=str(config)) as local:
        offsets = [local.publish_topic("changes", batch=b) for b in batches]
    subscriber.join(timeout=60)
    replay = remote.subscribe_collect("changes", from_offset=0)

live = received["live"]
published = pa.concat_tables(batches)
print(f"published at offsets {offsets}: {published.num_rows} events")
print(f"received live in the server process: {live.num_rows} events")
print(f"replayed from the backing table:     {replay.num_rows} events")

In [ ]:
assert offsets == [0, 1, 2]
assert not subscriber.is_alive()
assert live.select(["op", "key"]).equals(published)
assert replay.select(["op", "key"]).equals(published)
contracts.assert_close("production.events_received", live.num_rows)

The live subscriber and the replay agree because they are the same thing: `NOTIFY`
only says *go look*; what the subscriber delivers comes from replaying the backing
table from its offset. A subscriber in a process that missed a wake-up — or started
after the publish — reads the same rows on its next poll.

In [ ]:
postgres.cleanup()

## What this shows

* **One database is the system of record.** A server and an embedded engine configured
  onto the same Postgres see one catalog: what one registers or computes, the other
  reads, with no copy and no coordination beyond the database
  (Kleppmann 2017).
* **The broker adds no service.** Topics are tables in that database; `LISTEN`/`NOTIFY`
  is a wake-up, and the log a subscriber reads is the backing table — so delivery across
  processes is a replay, and a missed wake-up costs latency, never data
  (Kreps et al. 2011).
* **Deployment is configuration.** Nothing in the program changed between the
  single-process chapters and this one except the two stanzas above.

The next chapter operates this deployment: backing it up and restoring it, and reading
what it reports about its own health.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.
- Kreps, Jay, Narkhede, Neha, Rao, Jun (2011) *Kafka: A Distributed Messaging System for Log Processing* Proceedings of the NetDB Workshop on Networking Meets Databases.